<a href="https://colab.research.google.com/github/nagamamo/machine_learning_for_finance/blob/main/4_2_support_vector_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**4.2　サポートベクター回帰（SVR）とCAPM理論**

　サポートベクター分類は、連続変数を用いる回帰分析へ拡張することで、ファイナンス分析における応用範囲が拡大します。サポートベクター回帰も、線形カーネル関数を用いる線形回帰、RBFカーネル関数を用いる非線形回帰他、複数の手法があります。最小二乗法は、散布するデータの誤差が最小化される直線を当てはめるのに対し、線形サポートベクター回帰は、最も距離が近い2つのデータの距離を最大化する境界を設けます。それゆえ、最小二乗法が当てはめる直線がデータの外れ値の影響を受けるのに対し、線形サポートベクター回帰はこの影響を受けません。それゆえ、同じ直線を当てはめる線形回帰とサポートベクター回帰の推計結果は異なる結果が得られます。そしてRBFカーネル関数他を用いる非線形サポートベクター回帰の場合は、非線形な境界を当てはめる推計が可能となることで、曲線的な関係をもつデータの分析が可能となります。

　本例では、サポートベクター回帰のCAPM理論への応用事例です。第2章のラッソ回帰に用いたデータと同じデータを用いますので、推計結果の違いを比較してみてください。まずsklearnからライブラリをインポートします。4.1のサポートベクター分類では4行目はimport SVCでしたが、回帰の場合はimport SVRと記述します。

In [1]:
#[1]ライブラリのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

　次にオンラインストレージから、日経平均株価と日経主要自動車メーカー3社の株価終値データを取得します。第2セルの3-4行目では、この時系列データの日付をインデックス化し、5行目でデータを確認するコードが記述されています。3社の日次終値株価以外に、曜日ダミー変数、月次ダミー変数も収録されています。

In [ ]:
#[2]データの取得
url = "https://github.com/nagamamo/data/blob/main/5_capm_lasso_data.csv?raw=true"#オンラインストレージからデータを取得
data = pd.read_csv(url)#pandasデータフレームへデータ変換
data["Date"] = pd.to_datetime(data["Date"], errors='coerce')
df = data.set_index("Date")#日付情報をインデックス化
df.head()

　取得したデータは、日経平均株価(Nikkei)、トヨタ（Toyota)、ホンダ（Honda)の3種類の連続データです。これらのデータを、曜日ダミー変数、月次ダミー変数とともに、X：説明変数、y：被説明変数として定義します。

In [3]:
#[3]データの定義
X = df[['Nikkei','Mon','Tue','Wed','Thr','Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov']]
y = df['Toyota']

　続いてデータの標準化を行った上で学習を進めます。

In [ ]:
#[3] 標準化と学習
scaler_X = StandardScaler() #説明変数の標準化
scaler_y = StandardScaler() #被説明変数の標準化
X_scaled = scaler_X.fit_transform(X) #データの学習と変換
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1)).ravel() #データの学習と変換
model = SVR(kernel='linear', C=1.0, epsilon=0.1)#モデルの選択
model.fit(X_scaled, y_scaled)

続いて、CAPM実証モデルの係数値と定数項を算出します。ここでは標準化されたデータを用いて推計を行い、その上で推計結果を原データを用いる場合のスケールへ逆変換します。

In [5]:
#[4]係数値と切片の算出
w_scaled = model.coef_.ravel()  #係数値を1次元配列化
b_scaled = model.intercept_[0]

In [6]:
#[5]原データへの逆変換
mean_X, scale_X = scaler_X.mean_, scaler_X.scale_ #標準化前のデータへ逆変換
mean_y, scale_y = scaler_y.mean_, scaler_y.scale_
w_original = w_scaled * (scale_y / scale_X)#各変数の係数値を逆変換
b_original = mean_y + b_scaled * scale_y - np.sum(w_scaled * scale_y * mean_X / scale_X)#切片を逆変換

　最後に推計結果を表示します。これを見ると、第2章のラッソ回帰により得られた係数値とほぼ同じ推計結果を示しています。他方、コントール変数の曜日効果、月次効果は、ラッソ回帰ではその多くが変数削除されたのに対し、サポートベクター回帰では多かれ少なかれ影響力を残していることがわかります。

In [7]:
#[6]推計結果の表示
for i in range(16): #説明変数の採用数を（）に記述
    print(f"X{i+1}: {w_original[i]:.4f}")
print(f"Intercept:{b_original[0]:.4f}")

X1: 0.9139
X2: 0.0418
X3: 0.0860
X4: 0.0890
X5: -0.0236
X6: -0.0858
X7: -0.0490
X8: 0.0228
X9: -0.1314
X10: -0.1711
X11: -0.0639
X12: 0.0750
X13: -0.0855
X14: 0.0527
X15: -0.0742
X16: -0.0933
Intercept:-0.0122
